In [49]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Credit Risk Assessment

In [50]:
import pandas as pd
import numpy as np

import src.utils as utils

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Modeling

### Model Baseline

In [51]:
df_train = pd.read_parquet('input/df_train_02.parquet')
X_train = df_train.drop(columns=['loan_status'])
y_train = df_train['loan_status']

In [52]:
df_test = pd.read_parquet('input/df_test_02.parquet')
X_test = df_test.drop(columns=['loan_status'])
y_test = df_test['loan_status']

In [53]:
utils.skim_data(df_train)

Total duplicate rows: 0
DF shape: (190693, 23)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1291,0.68,"[6000, 9250, 15600, 15000, 1050]"
1,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
2,sub_grade,object,0.000,-,-,35,0.02,"[D3, C2, A5, C4, C5]"
3,emp_length,object,3.807,-,-,11,0.01,"[9 years, 10+ years, 4 years, 1 year, 8 years]"
4,home_ownership,object,0.000,-,-,6,0.00,"[OWN, RENT, MORTGAGE, OTHER, NONE]"
5,annual_inc,float64,0.000,0.0,0.0,15697,8.23,"[60000.0, 27600.0, 75000.0, 80000.0, 36000.0]"
6,verification_status,object,0.000,-,-,3,0.00,"[Verified, Not Verified, Source Verified]"
7,purpose,object,0.000,-,-,14,0.01,"[home_improvement, credit_card, debt_consolida..."
8,addr_state,object,0.000,-,-,50,0.03,"[NY, AZ, SC, MI, CT]"
9,dti,float64,0.000,0.0,0.128,3868,2.03,"[20.3, 10.87, 22.32, 10.74, 14.77]"


In [54]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier

dummy_classifier = DummyClassifier(strategy='most_frequent', random_state=29)
dummy_classifier.fit(X_train, y_train)
y_pred_dummy = dummy_classifier.predict(X_test)
y_pred_proba_dummy = dummy_classifier.predict_proba(X_test)[:, 1]
print("--- Dummy Classifier (Baseline) ---")
print(classification_report(y_test, y_pred_dummy))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_dummy):.4f}\n")

--- Dummy Classifier (Baseline) ---
              precision    recall  f1-score   support

           0       0.78      1.00      0.88     37267
           1       0.00      0.00      0.00     10406

    accuracy                           0.78     47673
   macro avg       0.39      0.50      0.44     47673
weighted avg       0.61      0.78      0.69     47673

ROC-AUC Score: 0.5000



### Fitting Models and Results

In [65]:
from sklearn.model_selection import GridSearchCV

def model_comparison_report(latest_model: GridSearchCV,
                            dummy_clf: DummyClassifier,
                            X_train, y_train,
                            X_test, y_test,
                            clf_name='RandomForest'):
    # get selected features
    best_pipeline = latest_model.best_estimator_
    preprocessor = best_pipeline.named_steps['preprocessing']

    if 'feature_selection' in best_pipeline.named_steps:
        feature_names = preprocessor.get_feature_names_out()
        feature_selector = best_pipeline.named_steps['feature_selection']
        selected_mask = feature_selector.get_support()
        selected_feature_names = feature_names[selected_mask].tolist()
        print(f'Selected features ({len(selected_feature_names)} total):')
        print(f'{selected_feature_names}')

    y_pred_test = latest_model.predict(X_test)
    y_proba_test = latest_model.predict_proba(X_test)[:, 1]
    y_dummy_proba_test = dummy_clf.predict_proba(X_test)[:, 1]
    y_pred_train = latest_model.predict(X_train)
    y_proba_train = latest_model.predict_proba(X_train)[:, 1]
    y_dummy_proba_train = dummy_clf.predict_proba(X_train)[:, 1]

    print("\n--- Perbandingan Skor ROC-AUC (Train Set) ---")
    print(f"Dummy Classifier (Baseline): {roc_auc_score(y_train, y_dummy_proba_train):.4f}")
    print(f"{clf_name} (Tuned): {roc_auc_score(y_train, y_proba_train):.4f}")

    print(f"\n--- Laporan Klasifikasi {clf_name} (Train Set) ---")
    print(classification_report(y_train, y_pred_train, target_names=['Good Loan (0)', 'Bad Loan (1)']))

    print("\n--- Perbandingan Skor ROC-AUC (Test Set) ---")
    print(f"Dummy Classifier (Baseline): {roc_auc_score(y_test, y_dummy_proba_test):.4f}")
    print(f"{clf_name} (Tuned): {roc_auc_score(y_test, y_proba_test):.4f}")

    print(f"\n--- Laporan Klasifikasi {clf_name} (Test Set) ---")
    print(classification_report(y_test, y_pred_test, target_names=['Good Loan (0)', 'Bad Loan (1)']))

#### Random Forest

In [56]:
latest_rf_grid = utils.load_model('models/rf_grid_2025_11_29_07_47_00.joblib')
model_comparison_report(latest_rf_grid, dummy_classifier, X_test, y_test)

Model loaded from: models/rf_grid_2025_11_29_07_47_00.joblib
Selected features (16 total):
['num__loan_amnt', 'num__sub_grade', 'num__annual_inc', 'num__verification_status', 'num__dti', 'num__delinq_2yrs', 'num__inq_last_6mths', 'num__open_acc', 'num__revol_bal', 'num__revol_util', 'num__total_acc', 'num__tot_cur_bal', 'num__total_rev_hi_lim', 'num__term', 'num__emp_length', 'num__credit_history_age']


ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: 'B5'

In [ ]:
latest_rf_grid = utils.load_model('models/rf_grid_2025_11_29_10_00_25.joblib')
model_comparison_report(latest_rf_grid, dummy_classifier, X_test, y_test)

Model loaded from: models/rf_grid_2025_11_29_10_00_25.joblib
Selected features (16 total):
['num__loan_amnt', 'num__sub_grade', 'num__annual_inc', 'num__verification_status', 'num__dti', 'num__delinq_2yrs', 'num__inq_last_6mths', 'num__open_acc', 'num__revol_bal', 'num__revol_util', 'num__total_acc', 'num__tot_cur_bal', 'num__total_rev_hi_lim', 'num__term', 'num__emp_length', 'num__credit_history_age']

--- Perbandingan Skor ROC-AUC ---
Dummy Classifier (Baseline): 0.5000
Random Forest (Tuned): 0.6905

--- Laporan Klasifikasi RandomForest (Test Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.81      0.89      0.85     37346
 Bad Loan (1)       0.42      0.27      0.33     10437

     accuracy                           0.76     47783
    macro avg       0.62      0.58      0.59     47783
 weighted avg       0.73      0.76      0.74     47783



In [ ]:
latest_rf_grid = utils.load_model('models/rf_grid_2025_11_29_10_54_02.joblib')
model_comparison_report(latest_rf_grid, dummy_classifier, X_test, y_test)

Model loaded from: models/rf_grid_2025_11_29_10_54_02.joblib
Selected features (12 total):
['num__loan_amnt', 'num__sub_grade', 'num__annual_inc', 'num__dti', 'num__open_acc', 'num__revol_bal', 'num__revol_util', 'num__total_acc', 'num__tot_cur_bal', 'num__total_rev_hi_lim', 'num__emp_length', 'num__credit_history_age']

--- Perbandingan Skor ROC-AUC ---
Dummy Classifier (Baseline): 0.5000
Random Forest (Tuned): 0.6932

--- Laporan Klasifikasi RandomForest (Test Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.86      0.67      0.75     37346
 Bad Loan (1)       0.34      0.61      0.44     10437

     accuracy                           0.66     47783
    macro avg       0.60      0.64      0.60     47783
 weighted avg       0.75      0.66      0.68     47783



#### AdaBoost

In [ ]:
latest_adb_grid = utils.load_model('models/adb_grid_2025_11_30_00_05_12.joblib')
model_comparison_report(latest_adb_grid, dummy_classifier, X_test, y_test)

Model loaded from: models/adb_grid_2025_11_30_00_05_12.joblib
Selected features (16 total):
['num__loan_amnt', 'num__sub_grade', 'num__annual_inc', 'num__verification_status', 'num__dti', 'num__delinq_2yrs', 'num__inq_last_6mths', 'num__open_acc', 'num__revol_bal', 'num__revol_util', 'num__total_acc', 'num__tot_cur_bal', 'num__total_rev_hi_lim', 'num__term', 'num__emp_length', 'num__credit_history_age']

--- Perbandingan Skor ROC-AUC ---
Dummy Classifier (Baseline): 0.5000
Random Forest (Tuned): 0.6583

--- Laporan Klasifikasi RandomForest (Test Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.88      0.46      0.60     37346
 Bad Loan (1)       0.29      0.79      0.42     10437

     accuracy                           0.53     47783
    macro avg       0.59      0.62      0.51     47783
 weighted avg       0.75      0.53      0.56     47783



In [ ]:
latest_adb_grid = utils.load_model('models/adb_grid_2025_11_30_00_27_43.joblib')
model_comparison_report(latest_adb_grid, dummy_classifier, X_test, y_test)

Model loaded from: models/adb_grid_2025_11_30_00_27_43.joblib
Selected features (16 total):
['num__loan_amnt', 'num__sub_grade', 'num__annual_inc', 'num__verification_status', 'num__dti', 'num__delinq_2yrs', 'num__inq_last_6mths', 'num__open_acc', 'num__revol_bal', 'num__revol_util', 'num__total_acc', 'num__tot_cur_bal', 'num__total_rev_hi_lim', 'num__term', 'num__emp_length', 'num__credit_history_age']

--- Perbandingan Skor ROC-AUC ---
Dummy Classifier (Baseline): 0.5000
Random Forest (Tuned): 0.6583

--- Laporan Klasifikasi RandomForest (Test Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.88      0.46      0.60     37346
 Bad Loan (1)       0.29      0.79      0.42     10437

     accuracy                           0.53     47783
    macro avg       0.59      0.62      0.51     47783
 weighted avg       0.75      0.53      0.56     47783



In [ ]:
latest_adb_grid = utils.load_model('models/adb_grid_2025_11_30_00_45_02.joblib')
model_comparison_report(latest_adb_grid, dummy_classifier, X_test, y_test)

Model loaded from: models/adb_grid_2025_11_30_00_45_02.joblib
Selected features (16 total):
['num__loan_amnt', 'num__sub_grade', 'num__annual_inc', 'num__verification_status', 'num__dti', 'num__delinq_2yrs', 'num__inq_last_6mths', 'num__open_acc', 'num__revol_bal', 'num__revol_util', 'num__total_acc', 'num__tot_cur_bal', 'num__total_rev_hi_lim', 'num__term', 'num__emp_length', 'num__credit_history_age']

--- Perbandingan Skor ROC-AUC ---
Dummy Classifier (Baseline): 0.5000
Random Forest (Tuned): 0.6686

--- Laporan Klasifikasi RandomForest (Test Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.80      0.96      0.87     37346
 Bad Loan (1)       0.48      0.14      0.22     10437

     accuracy                           0.78     47783
    macro avg       0.64      0.55      0.54     47783
 weighted avg       0.73      0.78      0.73     47783



In [ ]:
latest_adb_grid = utils.load_model('models/adb_grid_2025_11_30_04_35_32.joblib')
model_comparison_report(latest_adb_grid, dummy_classifier, X_test, y_test)

Model loaded from: models/adb_grid_2025_11_30_04_35_32.joblib
Selected features (16 total):
['num__loan_amnt', 'num__sub_grade', 'num__annual_inc', 'num__verification_status', 'num__dti', 'num__delinq_2yrs', 'num__inq_last_6mths', 'num__open_acc', 'num__revol_bal', 'num__revol_util', 'num__total_acc', 'num__tot_cur_bal', 'num__total_rev_hi_lim', 'num__term', 'num__emp_length', 'num__credit_history_age']

--- Perbandingan Skor ROC-AUC ---
Dummy Classifier (Baseline): 0.5000
Random Forest (Tuned): 0.7045

--- Laporan Klasifikasi RandomForest (Test Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.80      0.95      0.87     37346
 Bad Loan (1)       0.50      0.16      0.24     10437

     accuracy                           0.78     47783
    macro avg       0.65      0.56      0.56     47783
 weighted avg       0.74      0.78      0.73     47783



In [ ]:
latest_adb_grid = utils.load_model('models/adb_grid_2025_11_30_08_49_12.joblib')
model_comparison_report(latest_adb_grid, dummy_classifier, X_test, y_test)

Model loaded from: models/adb_grid_2025_11_30_08_49_12.joblib
Selected features (11 total):
['num__sub_grade', 'num__annual_inc', 'num__verification_status', 'num__dti', 'num__inq_last_6mths', 'num__revol_util', 'num__term', 'num__emp_length', 'num__initial_list_status', 'cat__1', 'cat__2']

--- Perbandingan Skor ROC-AUC ---
Dummy Classifier (Baseline): 0.5000
Random Forest (Tuned): 0.6658

--- Laporan Klasifikasi RandomForest (Test Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.88      0.51      0.64     37346
 Bad Loan (1)       0.30      0.75      0.43     10437

     accuracy                           0.56     47783
    macro avg       0.59      0.63      0.54     47783
 weighted avg       0.75      0.56      0.60     47783



In [ ]:
utils.skim_data(df_train)

Total duplicate rows: 0
DF shape: (191130, 23)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1291,0.68,"[6000, 9250, 15600, 15000, 1050]"
1,sub_grade,int8,0.000,0.0,2.275,35,0.02,"[17, 11, 4, 13, 14]"
2,home_ownership,object,0.000,-,-,6,0.00,"[OWN, RENT, MORTGAGE, OTHER, NONE]"
3,annual_inc,float64,0.002,0.0,0.0,15749,8.24,"[60000.0, 27600.0, 75000.0, 80000.0, 36000.0]"
4,verification_status,int64,0.000,0.0,34.692,3,0.00,"[2, 0, 1]"
5,purpose,object,0.000,-,-,14,0.01,"[home_improvement, credit_card, debt_consolida..."
6,addr_state,object,0.000,-,-,50,0.03,"[NY, AZ, SC, MI, CT]"
7,dti,float64,0.000,0.0,0.129,3868,2.02,"[20.3, 10.87, 22.32, 10.74, 14.77]"
8,delinq_2yrs,float64,0.013,0.0,83.96,23,0.01,"[0.0, 2.0, 1.0, 4.0, 3.0]"
9,inq_last_6mths,float64,0.013,0.0,47.937,27,0.01,"[2.0, 0.0, 1.0, 4.0, 3.0]"


In [ ]:
latest_adb_grid = utils.load_model('models/adb_grid_2025_11_30_10_56_56.joblib')
model_comparison_report(latest_adb_grid, dummy_classifier, X_test, y_test)

Model loaded from: models/adb_grid_2025_11_30_10_56_56.joblib
Selected features (26 total):
['num__loan_amnt', 'num__annual_inc', 'num__dti', 'num__delinq_2yrs', 'num__inq_last_6mths', 'num__open_acc', 'num__pub_rec', 'num__revol_bal', 'num__revol_util', 'num__total_acc', 'num__tot_cur_bal', 'num__total_rev_hi_lim', 'num__emp_length', 'num__credit_history_age', 'cat__home_ownership_MORTGAGE', 'cat__home_ownership_RENT', 'cat__verification_status_Not Verified', 'cat__verification_status_Source Verified', 'cat__verification_status_Verified', 'cat__term_ 36 months', 'cat__term_ 60 months', 'cat__initial_list_status_f', 'cat__initial_list_status_w', 'tar__0', 'tar__1', 'tar__2']

--- Perbandingan Skor ROC-AUC ---
Dummy Classifier (Baseline): 0.5000
Random Forest (Tuned): 0.6656

--- Laporan Klasifikasi RandomForest (Test Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.86      0.61      0.71     37267
 Bad Loan (1)       0.32      0.66      0.43     10

In [66]:
latest_adb_grid = utils.load_model('models/adb_random_2025_11_30_11_40_31.joblib')
model_comparison_report(latest_adb_grid,
                        dummy_classifier,
                        X_train=X_train, y_train=y_train,
                        X_test=X_test, y_test=y_test,
                        clf_name="AdaBoost")

Model loaded from: models/adb_random_2025_11_30_11_40_31.joblib

--- Perbandingan Skor ROC-AUC (Train Set) ---
Dummy Classifier (Baseline): 0.5000
AdaBoost (Tuned): 0.6774

--- Laporan Klasifikasi AdaBoost (Train Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.83      0.82      0.82    149069
 Bad Loan (1)       0.37      0.39      0.38     41624

     accuracy                           0.72    190693
    macro avg       0.60      0.60      0.60    190693
 weighted avg       0.73      0.72      0.73    190693


--- Perbandingan Skor ROC-AUC (Test Set) ---
Dummy Classifier (Baseline): 0.5000
AdaBoost (Tuned): 0.6747

--- Laporan Klasifikasi AdaBoost (Test Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.83      0.82      0.82     37267
 Bad Loan (1)       0.37      0.38      0.37     10406

     accuracy                           0.72     47673
    macro avg       0.60      0.60      0.60     47673
 weighted av